# Torsional buckling of ring-stiffened cylinders: Moore and Wescoat (1944), Table I

Moore and Wescoat [1] tested in torsion 24S-T aluminium-alloy cylinders of
mean diameter $D = 30.08$ in, overall length 36 in and sheet thickness
$t = 0.020$ in, without stiffeners (specimen 16) and with $1/2 \times 1/2$ in
solid ring stiffeners inside the skin (specimens 17, 18, 13 and 13A), and
gave in Table I (p. 25) the average observed shear-buckling stress and the
theoretical one, from Donnell's formula for hinged edges [2]. This
notebook computes the first buckling stress of the perfect cylinders with
the BFSC cylinder element of Sanders-type kinematics
(`bfsccylinder.sanders.BFSCCylinderSanders`) and compares it with both
columns of Table I. The stiffened specimens 19, 14, 20, 15 and 21 of the
same report (Table II) are the subject of the notebook
`moore1944_stiffened_cylinders.ipynb`.

Pages are the printed ones of [1] (pdf page = printed page + 2).

**References**

1. Moore, R. L., and Wescoat, C., Torsion tests of stiffened circular
   cylinders, NACA ARR 4E31 (Wartime Report W-89), 1944.
2. Donnell, L. H., Stability of thin-walled tubes under torsion, NACA
   Report 479, 1933 (reference 6 of [1]; formula quoted in the appendix of
   [1], p. 23).

In [1]:
import sys
import time
sys.path.append('.')

import numpy as np
import matplotlib.pyplot as plt

import moore1944_utils as mu

## Data

From [1]:

- material 24S-T, $E = 10.5 \times 10^6$ psi, $\nu = 1/3$ (p. 7, where
  also $G = 3.95 \times 10^6$ psi; $E/(2 (1 + \nu)) = 3.94 \times 10^6$
  psi is used here);
- mean diameter $D = 30.08$ in, $R = D/2 = 15.04$ in (Table I);
- end bulkheads: 3/4 in steel plates to which the shell is screwed through
  a washer strip (Fig. 1); the length between the centres of the
  bulkheads, $35 \tfrac{5}{8}$ in (Fig. 5), equal to the 35.62 in of
  specimen 16 in Table I, is the length $L$ of the models;
- rings: $1/2 \times 1/2$ in solid 24S-T bars inside the skin, pinned to
  it (p. 3, Fig. 1); stations from Fig. 5, measured from the centres of
  the bulkheads;
- the longitudinal seam of every specimen is covered by a stringer that
  is "not considered effective" for the ring-only specimens (p. 4,
  footnote); it is left out of the models.

The bays (in) between bulkhead and ring centres:

In [2]:
print('%-5s %-6s %s' % ('spec', 't (in)', 'bays between bulkhead and ring centres (in)'))
for s in ['16', '17', '18', '13', '13A']:
    sp = mu.SPECIMENS[s]
    print('%-5s %-6.3f %s' % (s, sp['t'], ' | '.join('%.4g' % b for b in sp['bays'])))

spec  t (in) bays between bulkhead and ring centres (in)
16    0.020  35.62
17    0.020  4.312 | 27 | 4.312
18    0.020  4.312 | 13.5 | 13.5 | 4.312
13    0.020  8.812 | 9 | 9 | 8.812
13A   0.020  4.312 | 4.5 | 9 | 9 | 4.5 | 4.312


## Membrane state and Donnell's formula

A torque $T$ on a perfect circular cylinder gives a uniform membrane shear
force in the skin, the same on every section $x$ = constant by
equilibrium,

$$
N_{xy} = \frac{T}{2 \pi R^2}, \qquad \tau = \frac{N_{xy}}{t}
= \frac{T}{2 \pi R^2 t} = \frac{24\, T_{\mathrm{ft\,lb}}}{\pi D^2 t}
$$

the last form being that of [1] (Fig. 5, note 6: $\tau = 0.422\, T$ for
$t = 0.020$ in). The rings rotate rigidly with their section and carry no
membrane force. The constant $N_{xy}$ is therefore the exact pre-buckling
membrane state, and the geometric stiffness matrix is that of a constant
$N_{xy}$ (`update_KG_constant_stress`); this is checked below against the
geometric stiffness matrix from a linear static analysis with a prescribed
end rotation.

The theoretical column of Table I is Donnell's formula for hinged ends [2],
as written in the appendix of [1], p. 23, valid for $L^2 t/D^3 < 5.2$:

$$
\tau_0 = \frac{E t^2}{(1 - \nu^2) L^2} \left[2.8 + \sqrt{2.6 + 1.4
\left(\sqrt{1 - \nu^2}\, \frac{L^2}{t D}\right)^{3/2}}\right]
$$

with $L$ the length of the buckled section, centre to centre of the rings
(Table I). Reproduced:

In [3]:
print('%-5s %8s %10s %12s %10s %8s' % ('spec', 'L (in)', 'L^2t/D^3', 'tau0 (psi)',
                                    'Table I', 'diff %'))
for s, (Lb, obs, th) in mu.TABLE_I.items():
    t = mu.SPECIMENS[s]['t']
    tau0 = mu.donnell_tau(t, Lb)
    print('%-5s %8.2f %10.4f %12.0f %10d %+8.1f' % (s, Lb, Lb**2*t/mu.D**3, tau0, th,
                                                   100*(tau0/th - 1)))

spec    L (in)   L^2t/D^3   tau0 (psi)    Table I   diff %
16       35.62     0.0009         1323       1325     -0.2
17       27.00     0.0005         1525       1520     +0.3
18       13.50     0.0001         2204       2210     -0.3
13        8.81     0.0001         2811       2870     -2.1
13A       9.00     0.0001         2775       2770     +0.2


The formula reproduces the column within 0.4 %, except specimen 13, 2.1 %
below the printed 2870 psi; the printed value could not be traced to a
different length (with $L = 8.81$ in the formula gives 2811 psi). Moore's
theoretical values are used as printed in the tables below.

## Model

`moore1944_utils.MooreSector` builds a periodic sector of the cylinder on
the sector model of `channel_cylinder_sector.py` (skin of
`BFSCCylinderSanders` elements, 10 degrees of freedom per node, cyclic
symmetry by a constraint basis $\boldsymbol{Z}_j$).

**Cyclic symmetry.** A ring-only cylinder is axisymmetric: its buckling
modes are $e^{i n \theta}$ around the circumference, one problem per wave
number $n$. The model is a thin slice of angle $\phi = 2\pi/n_s$ ($n_s$
slices, two elements across) whose edge $\theta = \phi$ is tied to the
edge $\theta = 0$ by $\boldsymbol{u}(\phi) = e^{i n \phi}
\boldsymbol{u}(0)$ (Bloch index $j = n$). The reduced matrices
$\boldsymbol{Z}_n^H \boldsymbol{K} \boldsymbol{Z}_n$ are Hermitian;
$\boldsymbol{K}$ is positive definite after the boundary conditions and the
lowest positive multipliers $\lambda$ of
$(\boldsymbol{K} + \lambda \boldsymbol{K}_G) \boldsymbol{x} = 0$ are the
largest eigenvalues $1/\lambda$ of $\boldsymbol{K}^{-1}(-\boldsymbol{K}_G)$.
With $\boldsymbol{K}_G$ of $N_{xy} = 1$ lb/in, $\lambda$ is the critical
$N_{xy}$ and $\tau_{cr} = \lambda/t$. Under torsion the multipliers of an
isotropic shell come in pairs $\pm\lambda$ (a helical mode and its mirror
image, here in the same $n$): the sign of the torque does not matter.

**Rings.** The finite element library has no beam element. A ring is a band
of skin elements of axial width $b_r = 1/2$ in centred at the ring station
(two elements across), whose laminate is the skin plus a layer of
thickness $d_r = 1/2$ in on the inner face, $-t/2 - d_r \le z \le -t/2$,
with $\boldsymbol{A}$, $\boldsymbol{B}$, $\boldsymbol{D}$ referred to the
middle surface of the skin (the base layer of the channel stringer of
`channel_cylinder_sector.TwoLayerSkin`, on the other face). With the
Poisson's ratio of the ring layer set to zero, $Q_{11} = Q_{22} = E$, the
band has the axial stiffness $E b_r d_r$ of the bar, its bending stiffness
about the skin middle surface, $E (I_r + b_r d_r e^2)$, $e = (t + d_r)/2$,
i.e. the bending of the bar normal to the skin including its eccentricity,
and, through the membrane stiffness of the band and the cubic in-plane
interpolation across it, the bending of the bar in the plane of the ring
section (about the radial axis). The continuous bond is the perfect bond
of the Kirchhoff kinematics across the 0.5 in layer: the normal of the skin
stays normal in the ring, an approximation for a layer 25 times thicker
than the skin; the torsional stiffness of the band, $G b_r d_r^3/3$, is 2.4
times the St Venant value of the square bar, $0.141\, G d_r^4$ (sensitivity
below). The 3/16 in pins of [1] are replaced by the continuous bond.

**Boundary conditions.** The ends are attached to rigid bulkheads; [1]
does not say whether they act as hinged or clamped edges (its theory
assumes hinged edges, and p. 9 estimates clamped edges to raise it by
10 to 12 %). The baseline is hinged: $u = v = w = 0$ and their derivatives
along the edge at both ends, $w_{,x}$ free; the sensitivity "clamped" adds
$w_{,x} = w_{,xy} = 0$. The sensitivity "rigid rings" adds $v = w = 0$ (and
$v_{,y} = w_{,y} = 0$) at the line of the ring centres.

## Mesh convergence

`dx` is the largest element length along $x$ between the ring bands,
`nslices` the number of slices around the circumference (element arc
$\pi R/n_s$) and `nring` the number of elements across a ring band. The
wave numbers $n = 4$ to 40 are scanned and the lowest $\tau_{cr}$ is
kept.

In [4]:
specs = ['16', '17', '18', '13', '13A']
NS = range(4, 41)


def lowest(spec, mesh, ends='hinged', rigid_rings=False, **kw):
    dx, nslices, nring = mesh
    m = mu.MooreSector(spec, dx, nslices=nslices, nring=nring, **kw)
    res = m.scan(NS, ends=ends, rigid_rings=rigid_rings)
    n = min(res, key=lambda j: res[j][0])
    return dict(model=m, n=n, tau=res[n][0]/m.t, mode=res[n][1],
                taus={j: res[j][0]/m.t for j in res})


meshes = [(1.0, 60, 2), (0.5, 120, 2), (0.35, 180, 2), (0.25, 240, 4)]
conv = {}
t0 = time.time()
for s in specs:
    for mesh in meshes:
        conv[s, mesh] = lowest(s, mesh)
print('%.0f s' % (time.time() - t0))
print('%-5s' % 'spec' + ''.join('%22s' % str(m) for m in meshes))
for s in specs:
    row = '%-5s' % s
    prev = None
    for mesh in meshes:
        r = conv[s, mesh]
        d = '' if prev is None else '%+.2f%%' % (100*(r['tau']/prev - 1))
        row += '%22s' % ('%.1f (n=%d) %s' % (r['tau'], r['n'], d))
        prev = r['tau']
    print(row)

59 s
spec           (1.0, 60, 2)         (0.5, 120, 2)        (0.35, 180, 2)        (0.25, 240, 4)
16           1412.2 (n=14)   1403.7 (n=14) -0.60%  1402.6 (n=14) -0.08%  1402.1 (n=14) -0.04%
17           1604.8 (n=15)   1593.4 (n=15) -0.71%  1591.6 (n=15) -0.11%  1590.8 (n=15) -0.05%
18           2279.4 (n=20)   2255.3 (n=20) -1.06%  2252.3 (n=20) -0.13%  2251.0 (n=20) -0.06%
13           2847.6 (n=23)   2813.8 (n=23) -1.19%  2810.1 (n=23) -0.13%  2808.6 (n=23) -0.06%
13A          2868.7 (n=23)   2833.4 (n=23) -1.23%  2828.9 (n=23) -0.16%  2827.1 (n=23) -0.06%


Each entry is $\tau_{cr}$ (psi), the wave number of the lowest mode and
the change from the previous mesh. The change at the last refinement is
below 1 % for every specimen (see the table), so the third mesh,
`(0.35, 180, 2)`, is used below.

In [5]:
mesh = (0.35, 180, 2)
base = {s: conv[s, mesh] for s in specs}
for s in specs:
    r = base[s]
    print('%-4s %6d reduced dofs, n = %2d, tau_cr = %.1f psi' % (
        s, (~r['model'].constrained_bc()).sum(), r['n'], r['tau']))

16     2036 reduced dofs, n = 14, tau_cr = 1402.6 psi
17     2076 reduced dofs, n = 15, tau_cr = 1591.6 psi
18     2116 reduced dofs, n = 20, tau_cr = 2252.3 psi
13     2116 reduced dofs, n = 23, tau_cr = 2810.1 psi
13A    2156 reduced dofs, n = 23, tau_cr = 2828.9 psi


## Cross-check: torque from a prescribed end rotation

For specimen 18 the end $x = L$ is rotated about the axis by a small angle
$\varphi$ ($v = R\varphi$, $v_{,y} = 0$ at its nodes, the other conditions
hinged), the linear static problem is solved for the axisymmetric index
$n = 0$, the torque $T$ is $R$ times the sum of the reactions on the
prescribed $v$ times $n_s$, and the geometric stiffness matrix is computed
from the displacements (`update_KG`, membrane forces of every integration
point from the laminate stiffnesses and the strains). The buckling torque
of the lowest wave number is compared with that of the constant-$N_{xy}$
matrix.

In [6]:
m = base['18']['model']
n18 = base['18']['n']
u, T = m.static_rotation(1e-3)
lam_rot, _ = m.buckling(n18, KG=m.KG_u(u))
T_rot = lam_rot[0]*T
tau_rot = T_rot/(2*np.pi*mu.R**2*m.t)
print('torsional stiffness T/phi = %.4g lb in/rad' % (T/1e-3))
print('prescribed rotation: T_cr = %.0f lb in = %.0f ft lb, tau = %.1f psi'
      % (T_rot, T_rot/12, tau_rot))
print('constant N_xy:       tau = %.1f psi, difference %+.3f %%'
      % (base['18']['tau'], 100*(tau_rot/base['18']['tau'] - 1)))

torsional stiffness T/phi = 4.955e+07 lb in/rad
prescribed rotation: T_cr = 64022 lb in = 5335 ft lb, tau = 2252.3 psi
constant N_xy:       tau = 2252.3 psi, difference +0.000 %


The two pre-buckling states give the same buckling stress: the constant
membrane shear is the exact pre-buckling state of the ring-stiffened
cylinder in torsion.

## Sensitivities

On the converged mesh: clamped ends; hinged ends with the axial
displacement $u$ free (only $v = w = 0$ and their derivatives along the
edge); rigid ring supports (hinged ends);
torsional stiffness of the ring band reduced to the St Venant value of the
square bar (shear modulus of the ring layer times $0.141 \cdot 3 = 0.42$);
Poisson's ratio of the ring layer $1/3$ instead of 0; Donnell-type
kinematics (`bfsccylinder.BFSCCylinder`) instead of Sanders-type.

In [7]:
sens = {
    'clamped': dict(ends='clamped'),
    'hinged, u free': dict(ends='hinged, u free'),
    'rigid rings': dict(rigid_rings=True),
    'ring GJ St Venant': dict(ring_gfac=0.42),
    'ring nu = 1/3': dict(ring_nu=1/3),
    'Donnell kinematics': dict(kinematics='donnell'),
}
sres = {}
for name, kw in sens.items():
    for s in specs:
        if s == '16' and name not in ('clamped', 'hinged, u free', 'Donnell kinematics'):
            continue
        sres[name, s] = lowest(s, mesh, **kw)
print('%-20s' % 'case' + ''.join('%18s' % s for s in specs))
print('%-20s' % 'hinged (baseline)' + ''.join('%18s' % ('%.0f (n=%d)' % (base[s]['tau'], base[s]['n'])) for s in specs))
for name in sens:
    row = '%-20s' % name
    for s in specs:
        r = sres.get((name, s))
        row += '%18s' % ('-' if r is None else '%.0f (n=%d) %+.1f%%' % (
            r['tau'], r['n'], 100*(r['tau']/base[s]['tau'] - 1)))
    print(row)

case                                16                17                18                13               13A
hinged (baseline)          1403 (n=14)       1592 (n=15)       2252 (n=20)       2810 (n=23)       2829 (n=23)
clamped              1405 (n=14) +0.2% 1592 (n=15) +0.0% 2252 (n=20) +0.0% 2819 (n=23) +0.3% 2829 (n=23) +0.0%
hinged, u free       1301 (n=12) -7.2% 1592 (n=15) -0.0% 2252 (n=20) -0.0% 2744 (n=22) -2.3% 2829 (n=23) -0.0%
rigid rings                          - 1592 (n=15) +0.0% 2254 (n=20) +0.1% 2813 (n=23) +0.1% 2832 (n=23) +0.1%
ring GJ St Venant                    - 1590 (n=15) -0.1% 2248 (n=20) -0.2% 2805 (n=23) -0.2% 2823 (n=23) -0.2%
ring nu = 1/3                        - 1592 (n=15) +0.0% 2253 (n=20) +0.0% 2810 (n=23) +0.0% 2830 (n=23) +0.0%
Donnell kinematics   1404 (n=14) +0.1% 1593 (n=15) +0.1% 2253 (n=20) +0.0% 2811 (n=23) +0.0% 2830 (n=23) +0.0%


## Comparison with Table I

$\tau_{FE}$ is the first buckling stress of the perfect cylinder with
hinged ends (baseline) and with clamped ends; Table I gives the average
observed stress and Moore's theoretical stress (Donnell, hinged, buckled
section $L$). Differences are relative to the Table I values.

In [8]:
rows = []
print('| spec | L (in) | n | tau FE hinged | tau FE clamped | rigid rings | observed | theory [1] '
      '| FE hinged/obs | FE hinged/theory | FE clamped/obs |')
print('|---|---|---|---|---|---|---|---|---|---|---|')
for s in specs:
    Lb, obs, th = mu.TABLE_I[s]
    h, c = base[s]['tau'], sres['clamped', s]['tau']
    rr = sres.get(('rigid rings', s))
    print('| %s | %.2f | %d | %.0f | %.0f | %s | %d | %d | %+.1f %% | %+.1f %% | %+.1f %% |' % (
        s, Lb, base[s]['n'], h, c, '-' if rr is None else '%.0f' % rr['tau'], obs, th,
        100*(h/obs - 1), 100*(h/th - 1), 100*(c/obs - 1)))

| spec | L (in) | n | tau FE hinged | tau FE clamped | rigid rings | observed | theory [1] | FE hinged/obs | FE hinged/theory | FE clamped/obs |
|---|---|---|---|---|---|---|---|---|---|---|
| 16 | 35.62 | 14 | 1403 | 1405 | - | 1110 | 1325 | +26.4 % | +5.9 % | +26.6 % |
| 17 | 27.00 | 15 | 1592 | 1592 | 1592 | 1280 | 1520 | +24.3 % | +4.7 % | +24.4 % |
| 18 | 13.50 | 20 | 2252 | 2252 | 2254 | 1710 | 2210 | +31.7 % | +1.9 % | +31.7 % |
| 13 | 8.81 | 23 | 2810 | 2819 | 2813 | 2310 | 2870 | +21.7 % | -2.1 % | +22.0 % |
| 13A | 9.00 | 23 | 2829 | 2829 | 2832 | 2570 | 2770 | +10.1 % | +2.1 % | +10.1 % |


**The same cases in panels.** The notebook `moore1944_ring_cylinders` of the panels package
models these cylinders with Ritz domains (`cylshell_clpt_sanders` skin, hinged ends with $u = v =
w = 0$, constant shear flow) and the rings as annular plate webs, `annular_clpt_donnell`, connected
to the inner surface of the skin (`'BFxcte'` with `face='lower'`). Its converged first buckling
stresses are 1402, 1574, 2207, 2732 and 2740 psi for specimens 16, 17, 18, 13 and 13A, i.e. 0.1,
1.1, 2.0, 2.8 and 3.2 % below the hinged values of this notebook. The difference grows with the
number of rings: there, rigid ring supports with $w_{,x}$ free lower the stresses by 0.8 to 3.0 %,
so that the rotational restraint of the skin by the rings matters, and the ring bands of this
notebook, with their Kirchhoff torsional stiffness, restrain it more than the annular webs. With
$u$ free at the loaded end panels gives 1352 psi for specimen 16 (2.2 % above Donnell's formula).

Torques: $T = \tau \pi D^2 t/24$ ft lb ($\tau = 0.422\,T$ for $t = 0.020$
in, Fig. 5 of [1]):

In [9]:
for s in specs:
    print('%-4s FE hinged %.0f ft lb, FE clamped %.0f ft lb, observed %.0f ft lb' % (
        s, mu.torque_ftlb(base[s]['tau'], 0.020), mu.torque_ftlb(sres['clamped', s]['tau'], 0.020),
        mu.torque_ftlb(mu.TABLE_I[s][1], 0.020)))

16   FE hinged 3322 ft lb, FE clamped 3328 ft lb, observed 2629 ft lb
17   FE hinged 3770 ft lb, FE clamped 3770 ft lb, observed 3032 ft lb
18   FE hinged 5335 ft lb, FE clamped 5335 ft lb, observed 4051 ft lb
13   FE hinged 6657 ft lb, FE clamped 6678 ft lb, observed 5472 ft lb
13A  FE hinged 6701 ft lb, FE clamped 6701 ft lb, observed 6088 ft lb


## Buckling modes

Normal displacement $w$ (real part, arbitrary scale) of the lowest mode
over the developed skin, hinged ends; the dashed lines are the ring
stations.

In [10]:
fig, axes = plt.subplots(len(specs), 1, figsize=(8, 2.2*len(specs)), sharex=True)
for ax, s in zip(axes, specs):
    r = base[s]
    m = r['model']
    theta, w = m.wave(r['mode'], r['n'])
    w = w/np.abs(w).max()
    ax.contourf(theta*mu.R, m.xlin, w, levels=21, cmap='RdBu_r')
    for xr in mu.ring_stations(mu.SPECIMENS[s]['bays']):
        ax.axhline(xr, color='k', ls='--', lw=0.8)
    ax.set_ylabel('$x$ (in)')
    ax.set_title('specimen %s: n = %d, tau = %.0f psi' % (s, r['n'], r['tau']), fontsize=9)
    amp = np.abs(w).max(axis=1)
    print('specimen %-3s: largest |w| at x = %.1f in; |w| > 0.5 max for %.1f <= x <= %.1f in'
          % (s, m.xlin[amp.argmax()], m.xlin[amp > 0.5].min(), m.xlin[amp > 0.5].max()))
axes[-1].set_xlabel(r'$R\theta$ (in)')
fig.tight_layout()
plt.show()

specimen 16 : largest |w| at x = 17.8 in; |w| > 0.5 max for 6.6 <= x <= 29.0 in


specimen 17 : largest |w| at x = 17.8 in; |w| > 0.5 max for 9.1 <= x <= 26.5 in
specimen 18 : largest |w| at x = 24.6 in; |w| > 0.5 max for 7.0 <= x <= 28.7 in


specimen 13 : largest |w| at x = 31.2 in; |w| > 0.5 max for 1.7 <= x <= 33.9 in


specimen 13A: largest |w| at x = 22.1 in; |w| > 0.5 max for 10.8 <= x <= 24.9 in


## Wave number

$\tau_{cr}$ against $n$ (baseline, hinged):

In [11]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for s in specs:
    taus = base[s]['taus']
    ax.plot(list(taus), list(taus.values()), '.-', label=s)
ax.set_xlabel('circumferential wave number $n$')
ax.set_ylabel(r'$\tau_{cr}$ (psi)')
ax.set_ylim(0, 8000)
ax.legend(title='specimen')
fig.tight_layout()
plt.show()

## Discussion

- **Specimen 16** (no rings) checks the model against Donnell's formula:
  with $u = v = w = 0$ at the ends the converged $\tau_{cr}$ is 5.9 %
  above Moore's theoretical value; with the axial displacement free at the
  ends ("hinged, u free") it is 1.7 % below it. The restraint of $u$ is
  the main difference between the hinged ends of the model and those of
  Donnell's solution; the bulkheads of the tests, screwed to the shell,
  restrain it.
- **Ends.** Clamping the ends ($w_{,x} = 0$) changes $\tau_{cr}$ by 0.3 %
  at most: the 10 to 12 % that [1] (p. 9) expects from clamped edges is not
  found once $u$ and $v$ are restrained at the ends (a sensitivity on $u$
  is in the table above).
- **Rings.** The ring bands behave as nearly rigid supports: rigid ring
  supports change $\tau_{cr}$ by 0.1 % at most, and the torsional
  stiffness and the Poisson's ratio of the ring layer by 0.2 % at most. The
  buckling stress follows the bay length, as Moore's use of Donnell's
  formula with the length "center to center of ring stiffeners" assumes,
  and the finite element values of specimens 17, 18, 13 and 13A are within
  $-2.1$ to $+2.1$ % of Moore's theoretical values. The modes (figure and printed
  extent, where $|w|$ exceeds half its largest value) occupy the 27 in bay
  of specimen 17, the two 13.5 in bays of specimen 18, the two middle 9 in
  bays of specimen 13A and all four bays of specimen 13, with the largest
  $|w|$ in an end bay; [1] reports the end sections of specimen 13 buckled.
- **Tests.** The observed stresses are 10 to 32 % below the finite element
  values of the perfect shell, as they are 7 to 23 % below Moore's theory
  (Table I, ratios 0.77 to 0.93): initial imperfections of the thin sheet
  and the onset of buckling read from the tests lower the measured values.